# Scenario 3: Freight economics

Business question: where does shipping cost eat a disproportionate share of
what the customer pays for the product itself?

One caveat that has to travel with every number here: the dataset has no
product cost, so freight ratio is not profit margin. A category with a 40%
freight ratio is a category where shipping adds 40% on top of the item
price. Whether that loses money is unknown; whether it hurts conversion is
the business argument.

Run from the project root:  python analysis/03_freight.py
Reads:  order_items_clean.csv, products_clean.csv, orders_clean.csv, customers_clean.csv, sellers_clean.csv
Writes: outputs/exports/freight_*.csv  and  outputs/figures/freight_*.png

In [1]:
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(20)

polars.config.Config

In [3]:
MIN_ITEMS_PER_GROUP = 200     # a category or state needs this many items to be compared fairly

## 1. Load

In [4]:
items = pl.read_csv("data/cleaned/order_items_clean.csv")
products = pl.read_csv("data/cleaned/products_clean.csv")
orders = pl.read_csv("data/cleaned/orders_clean.csv", try_parse_dates=True)
customers = pl.read_csv("data/cleaned/customers_clean.csv", schema_overrides={"customer_zip_code_prefix": pl.String})
sellers = pl.read_csv("data/cleaned/sellers_clean.csv", schema_overrides={"seller_zip_code_prefix": pl.String})

In [5]:
items.shape, products.shape

((112650, 7), (32951, 11))

## 2. Keep delivered orders only
A canceled order was quoted freight but never shipped it. Delivered orders
are the ones where the freight was actually paid and the goods actually moved.

In [6]:
items = items.join(orders.select("order_id", "order_status", "customer_id"), on="order_id", how="left")
items = items.filter(pl.col("order_status") == "delivered")
items.shape

(110197, 9)

## 3. Attach category, customer state, seller state
Three left joins, each against a table with one row per key, so no fan-out.

In [7]:
items = items.join(products.select("product_id", "product_category_name_english"), on="product_id", how="left")
items = items.join(customers.select("customer_id", "customer_state"), on="customer_id", how="left")
items = items.join(sellers.select("seller_id", "seller_state"), on="seller_id", how="left")
items.shape

(110197, 12)

In [8]:
items.select("product_category_name_english", "customer_state", "seller_state").null_count()

product_category_name_english,customer_state,seller_state
u32,u32,u32
0,0,0


## 4. The metric
freight_ratio = freight_value / price, per item.
When aggregating, use sum(freight) / sum(price), not the mean of the per-item
ratios. The mean of ratios lets a R$5 item with R$15 freight (ratio 3.0) pull
a whole category up; the ratio of sums weights every real (R$) equally.

In [9]:
items = items.with_columns(
    (pl.col("freight_value") / pl.col("price")).round(4).alias("freight_ratio"),
    (pl.col("freight_value") > pl.col("price")).alias("freight_exceeds_price"),
    (pl.col("seller_state") == pl.col("customer_state")).alias("same_state"),
)
items.select("price", "freight_value", "freight_ratio", "freight_exceeds_price").head(5)

price,freight_value,freight_ratio,freight_exceeds_price
f64,f64,f64,bool
58.9,13.29,0.2256,false
239.9,19.93,0.0831,false
199.0,17.87,0.0898,false
12.99,12.79,0.9846,false
199.9,18.14,0.0907,false


In [10]:
overall = pl.DataFrame({
    "metric": ["items", "total_price", "total_freight", "freight_ratio_overall",
               "median_item_freight_ratio", "share_items_freight_exceeds_price"],
    "value": [
        float(items.height),
        round(float(items["price"].sum()), 2),
        round(float(items["freight_value"].sum()), 2),
        round(float(items["freight_value"].sum() / items["price"].sum()), 4),
        round(float(items["freight_ratio"].median()), 4),
        round(float(items["freight_exceeds_price"].mean()), 4),
    ],
})
overall

metric,value
str,f64
"""items""",110197.0
"""total_price""",1.3221e7
"""total_freight""",2.1983e6
"""freight_ratio_overal…",0.1663
"""median_item_freight_…",0.2318
"""share_items_freight_…",0.0364


## 5. By product category
Only categories with at least 200 items are compared; the rest are listed
but excluded from the "worst" ranking.

In [11]:
by_category = (
    items.group_by("product_category_name_english")
    .agg(
        pl.len().alias("items"),
        pl.col("price").sum().round(2).alias("total_price"),
        pl.col("freight_value").sum().round(2).alias("total_freight"),
        pl.col("price").mean().round(2).alias("avg_item_price"),
        pl.col("freight_exceeds_price").mean().round(4).alias("share_freight_exceeds_price"),
    )
    .with_columns((pl.col("total_freight") / pl.col("total_price")).round(4).alias("freight_ratio"))
    .with_columns((pl.col("items") >= MIN_ITEMS_PER_GROUP).alias("enough_items"))
    .sort("freight_ratio", descending=True)
)
by_category.filter(pl.col("enough_items")).head(15)

product_category_name_english,items,total_price,total_freight,avg_item_price,share_freight_exceeds_price,freight_ratio,enough_items
str,u32,f64,f64,f64,f64,f64,bool
"""electronics""",2729,155043.93,45679.16,56.81,0.2202,0.2946,true
"""food_drink""",269,14942.88,4394.89,55.55,0.0818,0.2941,true
"""furniture_living_roo…",495,67350.7,17694.37,136.06,0.0141,0.2627,true
"""kitchen_dining_laund…",274,45531.98,11540.99,166.18,0.0219,0.2535,true
"""drinks""",361,21529.84,5441.27,59.64,0.0526,0.2527,true
"""office_furniture""",1668,268154.31,67057.05,160.76,0.0048,0.2501,true
"""food""",499,28731.15,7063.53,57.58,0.018,0.2458,true
"""furniture_decor""",8160,711927.69,168402.23,87.25,0.0392,0.2365,true
"""housewares""",6795,615628.69,142763.56,90.6,0.055,0.2319,true


In [12]:
by_category.filter(pl.col("enough_items")).tail(5)       # the cheapest-to-ship categories, for contrast

product_category_name_english,items,total_price,total_freight,avg_item_price,share_freight_exceeds_price,freight_ratio,enough_items
str,u32,f64,f64,f64,f64,f64,bool
"""home_appliances_2""",231,107953.95,10254.57,467.33,0.013,0.095,true
"""watches_gifts""",5859,1.1662e6,98156.14,199.04,0.0051,0.0842,true
"""small_appliances""",658,182754.12,15354.35,277.74,0.0137,0.084,true
"""fixed_telephony""",255,55315.21,4501.33,216.92,0.098,0.0814,true
"""agro_industry_and_co…",206,70566.1,5637.2,342.55,0.0583,0.0799,true


## 6. By price band
This is what supports a minimum-order-value recommendation. If freight is
60% of a R$20 item and 8% of a R$200 item, the fix is not the carrier.

In [13]:
items = items.with_columns(
    pl.when(pl.col("price") < 25).then(pl.lit("a. under 25"))
    .when(pl.col("price") < 50).then(pl.lit("b. 25 to 50"))
    .when(pl.col("price") < 100).then(pl.lit("c. 50 to 100"))
    .when(pl.col("price") < 200).then(pl.lit("d. 100 to 200"))
    .otherwise(pl.lit("e. 200 and up"))
    .alias("price_band")
)

In [14]:
by_price_band = (
    items.group_by("price_band")
    .agg(
        pl.len().alias("items"),
        pl.col("price").sum().round(2).alias("total_price"),
        pl.col("freight_value").sum().round(2).alias("total_freight"),
        pl.col("freight_value").mean().round(2).alias("avg_freight"),
        pl.col("freight_exceeds_price").mean().round(4).alias("share_freight_exceeds_price"),
    )
    .with_columns((pl.col("total_freight") / pl.col("total_price")).round(4).alias("freight_ratio"))
    .sort("price_band")
)
by_price_band

price_band,items,total_price,total_freight,avg_freight,share_freight_exceeds_price,freight_ratio
str,u32,f64,f64,f64,f64,f64
"""a. under 25""",13152,234599.78,181293.31,13.78,0.2647,0.7728
"""b. 25 to 50""",25092,963348.83,382633.09,15.25,0.0165,0.3972
"""c. 50 to 100""",32492,2.4302e6,580446.59,17.86,0.0026,0.2388
"""d. 100 to 200""",26429,3.7826e6,609101.46,23.05,0.0011,0.161
"""e. 200 and up""",13032,5.8107e6,444801.19,34.13,0.0,0.0765


## 7. By customer state, and same-state vs cross-state
Same-state shipping being much cheaper is the argument for regional
fulfilment: stock popular items closer to where they are bought.

In [15]:
by_state = (
    items.group_by("customer_state")
    .agg(
        pl.len().alias("items"),
        pl.col("price").sum().round(2).alias("total_price"),
        pl.col("freight_value").sum().round(2).alias("total_freight"),
        pl.col("freight_value").mean().round(2).alias("avg_freight"),
        pl.col("same_state").mean().round(4).alias("share_same_state_seller"),
    )
    .with_columns((pl.col("total_freight") / pl.col("total_price")).round(4).alias("freight_ratio"))
    .with_columns((pl.col("items") >= MIN_ITEMS_PER_GROUP).alias("enough_items"))
    .sort("freight_ratio", descending=True)
)
by_state

customer_state,items,total_price,total_freight,avg_freight,share_same_state_seller,freight_ratio,enough_items
str,u32,f64,f64,f64,f64,f64,bool
"""RR""",46,7057.47,1982.05,43.09,0.0,0.2808,false
"""MA""",800,117009.38,30794.17,38.49,0.0188,0.2632,true
"""RO""",273,45682.76,11283.24,41.33,0.0,0.247,true
"""AM""",163,22155.84,5429.63,33.31,0.0,0.2451,false
"""SE""",375,56574.19,13714.94,36.57,0.0,0.2424,true
"""PI""",523,84721.0,20457.19,39.12,0.0019,0.2415,true
"""TO""",310,48402.51,11604.86,37.44,0.0,0.2398,true
"""AC""",91,15930.97,3644.36,40.05,0.0,0.2288,false
"""PE""",1746,251889.49,57082.56,32.69,0.0137,0.2266,true


In [16]:
same_vs_cross = (
    items.group_by("same_state")
    .agg(
        pl.len().alias("items"),
        pl.col("price").sum().round(2).alias("total_price"),
        pl.col("freight_value").sum().round(2).alias("total_freight"),
        pl.col("freight_value").mean().round(2).alias("avg_freight"),
    )
    .with_columns((pl.col("total_freight") / pl.col("total_price")).round(4).alias("freight_ratio"))
    .with_columns(pl.when(pl.col("same_state")).then(pl.lit("same state")).otherwise(pl.lit("cross state")).alias("route"))
    .select("route", "items", "total_price", "total_freight", "avg_freight", "freight_ratio")
    .sort("route")
)
same_vs_cross

route,items,total_price,total_freight,avg_freight,freight_ratio
str,u32,f64,f64,f64,f64
"""cross state""",70331,9.0872e6,1.6621e6,23.63,0.1829
"""same state""",39866,4.1343e6,536174.09,13.45,0.1297


In [17]:
share_cross_state = 1 - items["same_state"].mean()
round(share_cross_state * 100, 1)        # % of delivered items that crossed a state line

63.8

## 8. Export

In [18]:
overall.write_csv("outputs/exports/freight_summary.csv")
by_category.write_csv("outputs/exports/freight_by_category.csv")
by_price_band.write_csv("outputs/exports/freight_by_price_band.csv")
by_state.write_csv("outputs/exports/freight_by_state.csv")
same_vs_cross.write_csv("outputs/exports/freight_same_vs_cross_state.csv")

## 9. Figures

In [19]:
top_cat = by_category.filter(pl.col("enough_items")).head(15)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_cat["product_category_name_english"].to_list(), (top_cat["freight_ratio"] * 100).to_list(), color="#8172B3")
ax.invert_yaxis()
ax.set_xlabel("Freight as % of item price")
ax.set_title("15 categories where shipping adds the most on top of price")
for i, (r, n) in enumerate(zip(top_cat["freight_ratio"].to_list(), top_cat["items"].to_list())):
    ax.text(r * 100 + 0.5, i, f"n={n:,}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("outputs/figures/freight_ratio_by_category.png", dpi=150)
plt.close()

In [20]:
fig, ax = plt.subplots(figsize=(8, 4.5))
labels = [b[3:] for b in by_price_band["price_band"].to_list()]      # strip the "a. " sort prefix
ax.bar(labels, (by_price_band["freight_ratio"] * 100).to_list(), color="#55A868")
ax.set_ylabel("Freight as % of item price")
ax.set_xlabel("Item price band (R$)")
ax.set_title("Cheap items carry the heaviest freight burden")
for i, v in enumerate(by_price_band["freight_ratio"].to_list()):
    ax.text(i, v * 100 + 1, f"{v*100:.0f}%", ha="center")
plt.tight_layout()
plt.savefig("outputs/figures/freight_ratio_by_price_band.png", dpi=150)
plt.close()

In [21]:
st = by_state.filter(pl.col("enough_items")).head(15)
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(st["customer_state"].to_list(), (st["freight_ratio"] * 100).to_list(), color="#DD8452")
ax.invert_yaxis()
ax.set_xlabel("Freight as % of item price")
ax.set_title("15 states with the heaviest freight burden (label: % of items from a same-state seller)")
for i, (r, s) in enumerate(zip(st["freight_ratio"].to_list(), st["share_same_state_seller"].to_list())):
    ax.text(r * 100 + 0.5, i, f"{s*100:.0f}% local", va="center", fontsize=8)
plt.tight_layout()
plt.savefig("outputs/figures/freight_ratio_by_state.png", dpi=150)
plt.close()

In [22]:
print("Scenario 3 done.")
print(f"  delivered items: {items.height:,}   overall freight ratio: {overall['value'][3]*100:.1f}%")
print(f"  items where freight > price: {overall['value'][5]*100:.1f}%")
print(f"  cross-state share: {share_cross_state*100:.1f}%   same-state ratio {same_vs_cross['freight_ratio'][1]*100:.1f}% vs cross-state {same_vs_cross['freight_ratio'][0]*100:.1f}%")

Scenario 3 done.
  delivered items: 110,197   overall freight ratio: 16.6%
  items where freight > price: 3.6%
  cross-state share: 63.8%   same-state ratio 13.0% vs cross-state 18.3%
